# T56-企业预警通新闻爬取

## 项目简介
本项目用于从企业预警通(www.qyyjt.cn)爬取新闻数据并存储到数据库。

---
## 第一章：环境配置与依赖导入

In [ ]:
# 第一章：环境配置与依赖导入
import pandas as pd
import numpy as np
import sqlalchemy
from datetime import datetime, timedelta
import json
import os
import warnings
warnings.filterwarnings('ignore')

# 导入配置
from config import DB_YQ_URL, DATA_DIR, OUTPUT_DIR, NEWS_CONFIG
from src import NewsSpider, DataStorage

print("依赖导入完成！")

---
## 第二章：数据库连接与爬虫初始化

In [ ]:
# 第二章：数据库连接与爬虫初始化
# 创建数据库连接
sql_engine = sqlalchemy.create_engine(DB_YQ_URL, poolclass=sqlalchemy.pool.NullPool)
cursor = sql_engine.connect()

print("数据库连接成功！")

# 初始化爬虫和存储模块
spider = NewsSpider()
storage = DataStorage(sql_engine)

print("模块初始化完成！")

---
## 第三章：API请求头配置

In [ ]:
# 第三章：API请求头配置
def get_request_headers(pcuss_token):
    """生成请求头"""
    return {
        "authority": "www.qyyjt.cn",
        "method": "POST",
        "path": "/getData.action",
        "scheme": "https",
        "Accept": "application/json",
        "Accept-Encoding": "gzip,deflate,br,zstd",
        "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
        "Client": "pc-web;pro",
        "Content-Type": "application/json;charset=UTF-8",
        "Dataid": "1447",
        "Origin": "https://www.qyyjt.cn",
        "Pcuss": pcuss_token,
        "Priority": "u=1,i",
        "Referer": "https://www.qyyjt.cn/publicOpinons/bondMarket",
        "Sec-Ch-Ua": '"Chromium";v="128","Not;A=Brand";v="24","MicrosoftEdge";v="128"',
        "Sec-Ch-Ua-Mobile": "?0",
        "Sec-Ch-Ua-Platform": "Windows",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "System": "new",
        "Terminal": "pc-web;pro",
        "User-Agent": "Mozilla/5.0(WindowsNT10.0;Win64;x64)AppleWebKit/537.36",
        "Ver": "20240903"
    }

# 示例：使用环境变量中的token
import os
pcuss_token = os.environ.get('QYYJT_PC_USS', '')
print(f"Token已配置: {'是' if pcuss_token else '否'}")

---
## 第四章：新闻数据爬取函数

In [ ]:
# 第四章：新闻数据爬取函数
import requests
import random
import time

def fetch_news_data(bdate, edate, topic_code='areaNews', sub_type='policyDynamic', headers=None):
    """获取新闻数据"""
    url = "https://www.qyyjt.cn/getData.action"
    
    data = {
        'bdate': bdate,
        'edate': edate,
        'pagesize': 15,
        'skipParam': 0,
        'subType': sub_type,
        'topicCode': topic_code
    }
    
    try:
        response = requests.post(url, headers=headers, json=data, timeout=30)
        result = response.json()
        news_list = result.get('data', {}).get('news', [])
        return news_list
    except Exception as e:
        print(f"请求失败: {e}")
        return []

def crawl_news_by_hour(start_date, end_date, headers):
    """按小时爬取新闻"""
    all_news = []
    current_date = start_date
    
    while current_date <= end_date:
        for hour in range(24):
            bdate = current_date.replace(hour=hour, minute=0, second=0).strftime('%Y%m%d%H%M%S')
            edate = current_date.replace(hour=hour, minute=59, second=59).strftime('%Y%m%d%H%M%S')
            
            news_list = fetch_news_data(bdate, edate, headers=headers)
            if news_list:
                all_news.extend(news_list)
            
            print(f"已处理: {current_date.strftime('%Y-%m-%d')} {hour}:00")
            time.sleep(random.uniform(0.5, 1))
        
        current_date += timedelta(days=1)
    
    return all_news

print("爬取函数定义完成！")

---
## 第五章：执行新闻爬取

In [ ]:
# 第五章：执行新闻爬取
# 设置爬取日期范围
start_datetime = datetime(2024, 10, 15)
end_datetime = datetime(2024, 10, 17)

print(f"爬取日期范围: {start_datetime.strftime('%Y-%m-%d')} 到 {end_datetime.strftime('%Y-%m-%d')}")

# 获取请求头
headers = get_request_headers(pcuss_token)

# 执行爬取
news_data = crawl_news_by_hour(start_datetime, end_datetime, headers)

print(f"\n共爬取到 {len(news_data)} 条新闻")

# 转换为DataFrame
if news_data:
    df_news = pd.DataFrame(news_data)
    print(f"\n数据形状: {df_news.shape}")
    print(f"\n列名: {df_news.columns.tolist()}")
    display(df_news.head())
else:
    df_news = pd.DataFrame()
    print("未获取到新闻数据")

---
## 第六章：数据存储到数据库

In [ ]:
# 第六章：数据存储到数据库
from sqlalchemy import inspect, text

if not df_news.empty:
    table_name = NEWS_CONFIG['table_name']
    
    # 转换数据类型
    df_news = df_news.astype(str)
    
    # 检查表是否存在
    inspector = inspect(sql_engine)
    table_exists = inspector.has_table(table_name)
    
    if table_exists:
        # 获取现有列
        existing_columns = inspector.get_columns(table_name)
        existing_columns_names = [col['name'] for col in existing_columns]
        
        # 添加缺失的列
        for col in df_news.columns:
            if col not in existing_columns_names:
                try:
                    with sql_engine.begin() as connection:
                        connection.execute(text(f"ALTER TABLE {table_name} ADD COLUMN {col} TEXT"))
                    print(f"添加列: {col}")
                except Exception as e:
                    print(f"添加列失败 {col}: {e}")
    
    # 保存数据
    try:
        with sql_engine.begin() as connection:
            df_news.to_sql(table_name, connection, if_exists='append', index=False)
        print(f"\n成功保存 {len(df_news)} 条数据到表 {table_name}")
    except Exception as e:
        print(f"保存失败: {e}")
else:
    print("没有数据需要保存")

---
## 第七章：数据验证与导出

In [ ]:
# 第七章：数据验证与导出
# 验证数据库中的数据
if storage.table_exists(NEWS_CONFIG['table_name']):
    verify_sql = f"SELECT COUNT(*) as cnt FROM {NEWS_CONFIG['table_name']}"
    result = pd.read_sql(verify_sql, cursor)
    print(f"表 {NEWS_CONFIG['table_name']} 中共有 {result['cnt'].iloc[0]} 条数据")
    
    # 显示最新数据
    sample_sql = f"SELECT * FROM {NEWS_CONFIG['table_name']} ORDER BY id DESC LIMIT 5"
    sample = pd.read_sql(sample_sql, cursor)
    print("\n最新5条数据:")
    display(sample)

# 导出到CSV
if not df_news.empty:
    output_file = os.path.join(OUTPUT_DIR, f'news_data_{datetime.now().strftime("%Y%m%d")}.csv')
    df_news.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n数据已导出到: {output_file}")

# 关闭连接
cursor.close()
print("\n数据库连接已关闭")